In [ ]:
import pandas as pd
import os
import numpy as np
import warnings
import statsmodels.api as sm
import statsmodels.tsa.stattools as ts
from statsmodels.tsa.api import VAR

import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split


In [ ]:
#-------------------------------------------------------#-------------------------------------------------------#
# The dataframe are created following the output of the LOBSTER data:
#     The variables for the message are (in order): Time: (expressed as second after 00:00); 
#                                   Event type: (1: submission of new l.o.: 2:Cancellation (partial del. of l.o.); 3:Deletion of l.o.; 4:Exec. of visible l.o.; 5: exec. of hidden l.o.; 6: cross trade; 7:trading halt indicator)
#                                   Order ID (0 for hidden order);
#                                   Size
#                                   Price: Dollar price times 10000. E.G. $91.14 -> 911400
#                                   Direction: -1 Sell, +1 Buy
#
#     VERY IMPORTANT: when event type = 7 it means the trades are halted, Price and Direction are setted to -1
#                     all the other to 0. When trading resume event type is 7 but PRICE = 1!!!!  
#-------------------------------------------------------#-------------------------------------------------------#

def dfmessage():
    warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

    data_cube_message = {}
    csv_directory_message = 'C:/Users/gabri/Desktop/Uni/Msc QF/2nd year/AlgoTrading/Paper/Data/Message' #Change with personal path
    for file_name in os.listdir(csv_directory_message):
        if file_name.endswith('.csv'):  # Only process CSV files
            
            day_label = file_name[:16]
            
            file_path = os.path.join(csv_directory_message, file_name)
            df = pd.read_csv(file_path, header=None, skiprows=1)
            df = df.drop(df.columns[-1], axis=1)
            
            df.columns = ["Sec. After MidN", "Event Type", "Order ID", "Size", "Price", "Direction"]
            df["Check"] = range(len(df))
            
            df["Direction"] = df["Direction"] * -1
            df["Signed Size"] = df["Direction"] * df["Size"]
            
            data_cube_message[day_label] = df
            print(f"**********Creating Dataframe for {day_label}**********", "\n")
    print("Dictionary for Trade Book created!!")
            
    return data_cube_message



dictmessage = dfmessage()


#-------------------------------------------------------#-------------------------------------------------------#
# The dataframe are created following the output of the LOBSTER data:
#     The variables for the order are (in order): Best Ask price
#                                                 Best Ask Size
#                                                 Best Bid Price
#                                                 Best Bid Size
#-------------------------------------------------------#-------------------------------------------------------#



def dforder():
    data_cube_orderbook = {}
    csv_directory_order = 'C:/Users/gabri/Desktop/Uni/Msc QF/2nd year/AlgoTrading/Paper/Data/OrderBook' #Change with personal path

    for file_name in os.listdir(csv_directory_order):
        if file_name.endswith('.csv'):  
            
            day_label = file_name[:16]
            
            file_path = os.path.join(csv_directory_order, file_name)
            df = pd.read_csv(file_path, header=None, skiprows=1)
            
            df.columns = ["BestAsk Price", "BestAsk Size", "BestBid Price", "BestBid Size"]
            


          
            data_cube_orderbook[day_label] = df
            print(f"**********Creating Dataframe for {day_label}**********", "\n")
    print("Dictionary for LOB created!!")
    return data_cube_orderbook

dictorder = dforder()

def dfjoiner():
    dictjoined = {}
    for i in dictmessage:
        res = pd.concat([dictmessage[i], dictorder[i]], axis=1)
        resf = res[res["Event Type"].isin([4, 5])]
        dictjoined[i] = resf
    return dictjoined

dictjoined = dfjoiner()


def addvariables():
    dictionary = {}
    for i in dictjoined:
        df = dictjoined[i]
        
        df["MidPrice"] = (df["BestAsk Price"] + df["BestBid Price"])/2
        df["MidPrice_t+1"] = df["MidPrice"].shift(-1)
        df["LN Return"] = np.log(df["MidPrice_t+1"]/df["MidPrice"])
        df["DepthImbalance"] = ((df["BestBid Size"] - df["BestAsk Size"]) / (df["BestBid Size"] + df["BestAsk Size"]))
        df["DI"] = np.where(df["DepthImbalance"] >= 0, '+DI', '-DI')
        df = df[:-1]
        dictionary[i] = df
    return dictionary

dbdata = addvariables()

####______________________________________________________########
# 
#      Calculate the transition prob from one state tuple
#      to another one.  
# 
####______________________________________________________########    


def trans_prob(x):
    df = x  

    df['State'] = list(zip(df['DI'], df['Direction']))
    df['Next_State'] = df['State'].shift(-1)

    df = df[:-1]

    transition_counts = df.groupby(['State', 'Next_State']).size().unstack(fill_value=0)
   
    transition_probs = transition_counts.div(transition_counts.sum(axis=1), axis=0)
    

    print("Transition Probabilities:")
    print(transition_probs)
    return transition_probs





In [ ]:
value_counts = dbdata["MSFT_2020-03-02_"]['DepthImbalance'].value_counts()
print(value_counts)

In [ ]:
dbdata["MSFT_2020-03-02_"][1000:10050]

In [ ]:
print(t.iloc[2:2])

In [ ]:
matrix = t.values.tolist() 
print(matrix)

In [ ]:
dfpossell = dbdata["MSFT_2020-03-02_"].loc[(dbdata["MSFT_2020-03-02_"]["DepthImbalance"] >= 0) & (dbdata["MSFT_2020-03-02_"]["Direction"] == -1)]
dfnegsell = dbdata["MSFT_2020-03-02_"].loc[(dbdata["MSFT_2020-03-02_"]["DepthImbalance"] < 0) & (dbdata["MSFT_2020-03-02_"]["Direction"] == -1)]
dfposbuy = dbdata["MSFT_2020-03-02_"].loc[(dbdata["MSFT_2020-03-02_"]["DepthImbalance"] >= 0) & (dbdata["MSFT_2020-03-02_"]["Direction"] == 1)]
dfnegbuy = dbdata["MSFT_2020-03-02_"].loc[(dbdata["MSFT_2020-03-02_"]["DepthImbalance"] < 0) & (dbdata["MSFT_2020-03-02_"]["Direction"] == 1)]




In [ ]:
m1 = np.mean(dfpossell["LN Return"])*10000
m2 = np.mean(dfnegsell["LN Return"])*10000
m3 = np.mean(dfposbuy["LN Return"])*10000
m4 = np.mean(dfnegbuy["LN Return"])*10000



print(m1)
print("\n")
print(m2)
print("\n")
print(m3)
print("\n")
print(m4)
print("\n")


In [ ]:
#-------------------------------------------------------#-------------------------------------------------------#
#       Recursion to find the Q, one can find the immediate rewards as the mean of the sliced dataframe
#       calculated above. 
#
#-------------------------------------------------------#-------------------------------------------------------#

g = 0.9
def permanent_price_impact():
    R_negDI_sell = -1
    R_posDI_sell = -0.5
    R_negDI_buy =  0.5
    R_posDI_buy = 1

    R = [R_negDI_sell, R_posDI_sell, R_negDI_buy, R_posDI_buy]

    gamma = g

    T_matrix = trans_prob(dbdata["MSFT_2020-03-02_"])
    T_matrix = T_matrix.values.tolist()

    Q_posDI_sell = 0
    Q_posDI_buy = 0
    Q_negDI_sell = 0
    Q_negDI_buy = 0

    Q = [Q_posDI_sell, Q_posDI_buy, Q_negDI_sell, Q_negDI_buy]
    Q_values = [Q.copy()]  # Store initial Q values

    for i in range(100):
        Q_posDI_sell = R_posDI_sell + gamma * (T_matrix[0][0] * Q[0] +
                                               T_matrix[0][1] * Q[1] +
                                               T_matrix[0][2] * Q[2] +
                                               T_matrix[0][3] * Q[3])

        Q_posDI_buy = R_posDI_buy + gamma * (T_matrix[1][0] * Q[0] +
                                             T_matrix[1][1] * Q[1] +
                                             T_matrix[1][2] * Q[2] +
                                             T_matrix[1][3] * Q[3])

        Q_negDI_sell = R_negDI_sell + gamma * (T_matrix[2][0] * Q[0] +
                                               T_matrix[2][1] * Q[1] +
                                               T_matrix[2][2] * Q[2] +
                                               T_matrix[2][3] * Q[3])

        Q_negDI_buy = R_negDI_buy + gamma * (T_matrix[3][0] * Q[0] +
                                             T_matrix[3][1] * Q[1] +
                                             T_matrix[3][2] * Q[2] +
                                             T_matrix[3][3] * Q[3])
        Q = [Q_posDI_sell, Q_posDI_buy, Q_negDI_sell, Q_negDI_buy]
        Q_values.append(Q.copy())  # Store Q values at each iteration

    return Q_values



# Calculate the differences between consecutive iterations
def calculate_differences(Q_values):
    differences = []
    for i in range(1, len(Q_values)):
        diff = [abs(Q_values[i][j] - Q_values[i-1][j]) for j in range(len(Q_values[i]))]
        
        differences.append(diff)
    
    
    return differences

def plot_convergence(differences):
    plt.figure(figsize=(10, 6))
    for j in range(len(differences[0])):
        plt.plot([diff[j] for diff in differences], label=f'Q[{j}]')
    plt.yscale('log')
    plt.title(f'Convergence of Q Values with gamma: {g}')
    plt.xlabel('Iteration')
    plt.ylabel('Difference (log scale)')
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 6))
    for j in range(len(differences[0])):
        plt.plot([diff[j] for diff in differences], label=f'Q[{j}]')
    plt.title(f'Convergence of Q Values with gamma: {g}')
    plt.xlabel('Iteration')
    plt.ylabel('Difference')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Plot the change list
    change = [sum(diff) for diff in differences]
    plt.figure(figsize=(10, 6))
    plt.plot(change, label='Total Change')
    plt.yscale('log')
    plt.title(f'Total Change in Q Values with gamma: {g}')
    plt.xlabel('Iteration')
    plt.ylabel('Total Change (log scale)')
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.plot(change, label='Total Change')
    plt.title(f'Total Change in Q Values with gamma: {g}')
    plt.xlabel('Iteration')
    plt.ylabel('Total Change')
    plt.legend()
    plt.grid(True)
    plt.show()

# Example usage
Q_values = permanent_price_impact()
differences = calculate_differences(Q_values)
plot_convergence(differences)



In [ ]:



def create_lagged_features(data, lags):
    for lag in lags:
        data[f"DepthImbalance_lag_{lag}"] = data["DepthImbalance"].shift(lag)
    return data

# Define the lags you want to create
lags = [1, 2, 3]  # You can adjust this based on your needs
xgdata = create_lagged_features(dbdata["MSFT_2020-03-02_"], lags)

# Drop rows with NaN values (due to lagging)
xgdata.dropna(inplace=True)

# Define features and target variable
X = xgdata[[f"DepthImbalance_lag_{lag}" for lag in lags]]
y = xgdata["Direction"].map({-1: 0, 1: 1})  # Predicting Direction


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

params = {
    "objective": "binary:logistic",  # Problema di classificazione binaria
    "eval_metric": "logloss",       # Log loss come metrica di valutazione
    "eta": 0.1,                     # Learning rate
    "use_label_encoder": True,
    "max_depth": 6,                 # Profondit? massima degli alberi
    "subsample": 0.8,               # Percentuale di dati usata per ogni albero
    "colsample_bytree": 0.8,        # Percentuale di feature usata per ogni albero
    "seed": 42                      # Riproducibilit?
}

# Train the XGBoost model
model = XGBClassifier(params=params)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(classification_report(y_test, y_pred))

# Sample every 100th value of the test set
sample_indices = np.arange(0, len(y_test), 500)
y_test_sampled = y_test.iloc[sample_indices]
y_pred_sampled = y_pred[sample_indices]

# Plotting the predictions and actual values
plt.figure(figsize=(12, 6))

# Points where actual equals predicted (correct predictions)
correct_indices = sample_indices[y_test_sampled == y_pred_sampled]
plt.scatter(correct_indices, y_test_sampled[y_test_sampled == y_pred_sampled], label="Correct Prediction", color="green", marker="*", s=100)

# Points where actual does not equal predicted (incorrect predictions)
incorrect_indices = sample_indices[y_test_sampled != y_pred_sampled]
plt.scatter(incorrect_indices, y_pred_sampled[y_test_sampled != y_pred_sampled], label="Predicted (Incorrect)", color="red", marker="x", s=100)

plt.title("Actual vs Predicted Direction (Sampled Every 500 Points)")
plt.xlabel("Index")
plt.ylabel("Direction")
plt.legend()
plt.grid(True)
plt.show()